# PHASE-13 PART-1D-B — PREDICTED-REGION SEGMENTATION ANALYSIS (REVIEWER)

**READ-ONLY.** This notebook performs **no inference**, constructs **no model**, requests **no CUDA**,
creates **no optimizer**, and **never touches TEST**. It reads only the frozen Part-1D-B tables and
audit artifacts produced by the authorized Part-1D-B inference run.

- run id: `phase13_part1d_b_predicted_region_clean_v3`
- manifest `phase4_v3` · taxonomy `kub_size_clean_v3_v1` · TEST **LOCKED**
- primary variant: **`PREDICTED_REGION_FLOAT_GLOBAL_RETAINED`**
- comparators: `GLOBAL_NATIVE` (reproduced in Part-1D-B) and `ORACLE_FLOAT_GLOBAL_RETAINED`
  (frozen Phase-12 authoritative table — **not** rerun)

> **Claim discipline.** This notebook reports *descriptive* gain/loss, retention, recovery/loss and
> cross-architecture consistency/heterogeneity. It does **not** issue `GO`, `REVISE_LOCALIZER` or
> `NO_GO`, does not declare a winner / best model / superiority, and makes no statistical or clinical
> superiority claim. The Part-1E-B supervisor gate comes later.

In [1]:
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 200)

ROOT = Path(r'D:\Softfors\jup')
TABLES = ROOT / 'reports' / 'tables'
AUDITS = ROOT / 'reports' / 'audits'


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for c in iter(lambda: f.read(1 << 20), b''):
            h.update(c)
    return h.hexdigest()


def show(df, n=None):
    d = df if n is None else df.head(n)
    print(d.to_string(index=False))
    print(f'[rows shown {len(d)} / {len(df)}]')


case = pd.read_csv(TABLES / 'phase13_part1d_predicted_region_case_model_metrics.csv')
comp = pd.read_csv(TABLES / 'phase13_part1d_predicted_region_component_model_metrics.csv')
summary = pd.read_csv(TABLES / 'phase13_part1d_predicted_region_model_summary.csv')
repro = pd.read_csv(TABLES / 'phase13_part1d_b_global_reproduction_gate.csv')
cohort = pd.read_csv(TABLES / 'phase13_part1d_predicted_region_cohort_summary.csv')
link = pd.read_csv(TABLES / 'phase13_part1d_localization_segmentation_linkage.csv')
cross = pd.read_csv(TABLES / 'phase13_part1d_cross_architecture_summary.csv')
gate = json.loads((AUDITS / 'phase13_regional_integration_gate_criteria_frozen.json').read_text('utf-8'))
r1 = json.loads((AUDITS / 'phase13_part1d_predicted_region_protocol_frozen_r1.json').read_text('utf-8'))
completion = json.loads((AUDITS / 'phase13_part1d_b_predicted_region_evaluation.json').read_text('utf-8'))
print('loaded frozen Part-1D-B artifacts')

loaded frozen Part-1D-B artifacts


## 2. R1 protocol identity

The Part-1D-B inference executed the **frozen R1** predicted-region protocol. The R1 revision repaired
square/padding crop-restore semantics **before** any inference (material pre-inference implementation
repair); no protocol semantics changed.

In [2]:
print('R1 artifact          :', r1['artifact'])
print('R1 status            :', r1['status'])
print('R1 revision          :', r1.get('revision'))
print('primary variant      :', r1['primary_variant'])
print('Part-1D helper SHA   :', r1['static_preflight']['module_sha256'])
print('static tests passed  :', r1['static_preflight']['tests_passed'])
print('R1 protocol SHA-256  :', sha256_file(AUDITS / 'phase13_part1d_predicted_region_protocol_frozen_r1.json'))

R1 artifact          : PHASE13_PART1D_PREDICTED_REGION_PROTOCOL_FROZEN_R1
R1 status            : SUPERVISOR_FROZEN_PART1D_R1_READY_FOR_INFERENCE
R1 revision          : R1
primary variant      : PREDICTED_REGION_FLOAT_GLOBAL_RETAINED
Part-1D helper SHA   : 8cd7e39814a049a025f706c57ccea0b535a230908a177690d6289318234c6284
static tests passed  : 31
R1 protocol SHA-256  : 9089b948ef7563252963bf68d0e0b57a9e891c5e913f54a315bce94c98207eb8


## 3. Prospective gate-criteria identity

The gate criteria were frozen in Part-1E-A **before** any Part-1D-B result existed. They are not
edited by this phase; Part-1D-B only generates evidence for later Part-1E-B adjudication.

In [3]:
print('gate artifact                  :', gate['artifact'])
print('gate status                    :', gate['status'])
print('part1d_b_results_inspected     :', gate['part1d_b_results_inspected'])
print('numerical_thresholds_introduced:', gate['numerical_thresholds_introduced'])
print('gate_state                     :', gate['gate_state'])
print('gate dimensions                :', list(gate['dimensions']))
print('gate outcomes                  :', list(gate['outcomes']))
print('gate criteria SHA-256          :', sha256_file(AUDITS / 'phase13_regional_integration_gate_criteria_frozen.json'))

gate artifact                  : PHASE13_REGIONAL_INTEGRATION_GATE_CRITERIA_FROZEN
gate status                    : PROSPECTIVE_GATE_CRITERIA_FROZEN_BEFORE_PART1D_B_RESULTS
part1d_b_results_inspected     : NO
numerical_thresholds_introduced: 0
gate_state                     : NOT_DECIDED
gate dimensions                : ['A', 'B', 'C', 'D']
gate outcomes                  : ['GO', 'REVISE_LOCALIZER', 'NO_GO', 'CHECK_REQUIRED']
gate criteria SHA-256          : 342a8bcac2c87bf0b85f7bbd963179e74f029825c0434cf527fbd939d2d13c21


## 4. Population and checkpoint bindings

In [4]:
print('VAL FULL_FIELD cases      :', case['sample_id'].nunique())
print('case-model rows           :', len(case))
print('unique sample x model     :', case[['sample_id', 'model']].drop_duplicates().shape[0])
print('native components         :', comp[['sample_id', 'native_component_id']].drop_duplicates().shape[0])
print('component-model rows      :', len(comp))
print('unique sample/comp/model  :', comp[['sample_id', 'native_component_id', 'model']].drop_duplicates().shape[0])
print('model summary rows        :', len(summary))
print()
print('size strata (component level):')
print(comp.drop_duplicates(['sample_id', 'native_component_id'])['size_stratum'].value_counts().to_string())
print()
print('proposal counts (frozen Part-1C):', sorted(case['proposal_count'].unique()))
print('total frozen proposals           :', int(case.drop_duplicates('sample_id')['proposal_count'].sum()))
print()
for k, v in completion['bindings']['checkpoints'].items():
    print(f'checkpoint {k:<16} {v["sha"]}  best_epoch={v["historical_best_epoch"]}')

VAL FULL_FIELD cases      : 103
case-model rows           : 412
unique sample x model     : 412
native components         : 152
component-model rows      : 608
unique sample/comp/model  : 608
model summary rows        : 4

size strata (component level):
size_stratum
TINY      72
MEDIUM    36
SMALL     28
LARGE     16

proposal counts (frozen Part-1C): [np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
total frozen proposals           : 494

checkpoint unet             7b955ba86464a2fec2d569ecb6124de1fa9a4e66d0b5d58b267be9a280b4c2c0  best_epoch=44
checkpoint attention_unet   a49b19d66e43b57ed082ea6efc0ac10bccf444277abd3540dc68612bf30b88b3  best_epoch=54
checkpoint unetpp           ec7eb13f32e43af27eab7fc617bd1a2a8fcce1d6651a1b4a3653f3e4d1283269  best_epoch=70
checkpoint deeplabv3plus    f519027ba1a3a4d6212d83fecd4379f61f71bc05b0751924b9660da08d597dad  best_epoch=52


## 5. Global reproduction gate

Stage G reproduced `GLOBAL_NATIVE` for all four architectures before any regional crop inference.
The integrity criterion is the **per-case native Dice** difference against the frozen Phase-12
`GLOBAL_NATIVE` rows; the frozen tolerance is **1e-4** (max absolute). Secondary reproduction fields
(IoU / precision / recall) are reported descriptively — **no new pass threshold is invented** for them.

In [5]:
print(repro[['model', 'rows_matched', 'max_abs_dice_diff', 'mean_abs_dice_diff',
             'mean_dice_new', 'mean_dice_phase12', 'mean_abs_iou_diff', 'max_abs_iou_diff',
             'mean_abs_precision_diff', 'mean_abs_recall_diff', 'pass']].to_string(index=False))
print()
print('ALL FOUR PASS :', bool(repro['pass'].all()))
print('tolerance     : 1e-4 (max abs case Dice)')

          model rows_matched  max_abs_dice_diff  mean_abs_dice_diff  mean_dice_new  mean_dice_phase12  mean_abs_iou_diff  max_abs_iou_diff  mean_abs_precision_diff  mean_abs_recall_diff  pass
           UNet      103/103       5.551115e-17        4.614701e-18       0.594180           0.594180       7.780993e-18      5.551115e-17             1.495567e-17          1.573041e-17  True
Attention U-Net      103/103       1.110223e-16        5.524168e-18       0.629805           0.629805       6.736790e-18      7.632783e-17             9.229403e-18          1.839144e-17  True
         UNet++      103/103       1.110223e-16        4.715753e-18       0.640516           0.640516       5.658904e-18      5.551115e-17             1.023992e-17          1.185675e-17  True
     DeepLabV3+      103/103       1.110223e-16        6.669422e-18       0.560251           0.560251       9.465190e-18      8.326673e-17             1.232833e-17          1.333884e-17  True

ALL FOUR PASS : True
tolerance     : 1e

## 6. Predicted-region fusion semantics

```
P(x) = R(x)   if U_pred(x)
     = G(x)   otherwise
```

- regional crops compose by **maximum regional probability only**
- `U_pred` = union of the clipped native frozen Part-1C proposal rectangles (never GT-derived)
- **regional REPLACEMENT** inside the support; `max(G, R)` inside the support is **forbidden**
- threshold **0.5** applied only after complete native fusion
- zero-proposal image → `P = G` exactly
- regional crop input/output uses the **R1** square/padding path
  (valid → zero-pad to full `crop_side` → 512 BILINEAR → model → sigmoid → 512 → full square → strip pad → native clip)

In [6]:
print('regional crop evaluations per model (frozen):', 494)
print('expected total 494 x 4                       :', 494 * 4)
print('recorded in completion audit                 :', completion['regional_crop_evaluations'])
print('regional_crop_evaluations == 1976            :', completion['regional_crop_evaluations'] == 1976)
print('zero-proposal cases (P = G exactly)          :', int((case.drop_duplicates('sample_id')['proposal_count'] == 0).sum()))

regional crop evaluations per model (frozen): 494
expected total 494 x 4                       : 1976
recorded in completion audit                 : 1976
regional_crop_evaluations == 1976            : True
zero-proposal cases (P = G exactly)          : 0


## 7. GLOBAL / ORACLE / PREDICTED macro results

Descriptive macro means over the 103 VAL FULL_FIELD cases. ORACLE values come from the **frozen
Phase-12 authoritative table** (no oracle rerun). **No winner, no rank.**

In [7]:
cols = ['model_display', 'global_macro_dice', 'oracle_macro_dice', 'predicted_macro_dice',
        'global_macro_iou', 'oracle_macro_iou', 'predicted_macro_iou',
        'global_macro_precision', 'oracle_macro_precision', 'predicted_macro_precision',
        'global_macro_recall', 'oracle_macro_recall', 'predicted_macro_recall']
show(summary[cols])

  model_display  global_macro_dice  oracle_macro_dice  predicted_macro_dice  global_macro_iou  oracle_macro_iou  predicted_macro_iou  global_macro_precision  oracle_macro_precision  predicted_macro_precision  global_macro_recall  oracle_macro_recall  predicted_macro_recall
           UNet           0.594180           0.724722              0.302314          0.476100          0.586966             0.198549                0.693919                0.670271                   0.234969             0.653931             0.859321                0.672113
Attention U-Net           0.629805           0.699595              0.308619          0.503956          0.559842             0.203629                0.670962                0.659250                   0.243961             0.701770             0.836333                0.688298
         UNet++           0.640516           0.726679              0.311355          0.521901          0.587570             0.206920                0.736065                0.6860

## 8. Dice oracle-benefit retention

`DELTA_ORACLE_DICE = D_O - D_G` · `DELTA_PRED_DICE = D_P - D_G` ·
`RETENTION_DICE = (D_P - D_G)/(D_O - D_G)` when the denominator > 0.
**Never clipped**; negative / 0..1 / >1 preserved. `ORACLE_GAP_DICE = D_P - D_O`.
Label: `PRACTICAL_GT_FREE_ORACLE_BENEFIT_RETENTION` — a descriptive retention quantity, not a gate cutoff.

In [8]:
show(summary[['model_display', 'global_macro_dice', 'oracle_macro_dice', 'predicted_macro_dice',
              'DELTA_ORACLE_DICE', 'DELTA_PRED_DICE', 'RETENTION_DICE', 'ORACLE_GAP_DICE']])
print()
print('denominator invalid (RETENTION_DICE is None) :',
      summary['RETENTION_DICE'].isna().sum())

  model_display  global_macro_dice  oracle_macro_dice  predicted_macro_dice  DELTA_ORACLE_DICE  DELTA_PRED_DICE  RETENTION_DICE  ORACLE_GAP_DICE
           UNet           0.594180           0.724722              0.302314           0.130542        -0.291866       -2.235800        -0.422408
Attention U-Net           0.629805           0.699595              0.308619           0.069790        -0.321186       -4.602209        -0.390976
         UNet++           0.640516           0.726679              0.311355           0.086164        -0.329161       -3.820171        -0.415324
     DeepLabV3+           0.560251           0.750372              0.277292           0.190121        -0.282959       -1.488308        -0.473080
[rows shown 4 / 4]

denominator invalid (RETENTION_DICE is None) : 0


## 9. IoU / object-recall / FN-reduction retention

All three use `(P - G)/(O - G)`-style gain retention with `N/A` on an invalid denominator.
**An invalid denominator is reported as `N/A` — never a fabricated zero.**

In [9]:
show(summary[['model_display', 'iou_gain_retention', 'object_recall_gain_retention',
              'fn_reduction_retention', 'global_object_recall', 'oracle_object_recall',
              'predicted_object_recall']])

  model_display  iou_gain_retention  object_recall_gain_retention  fn_reduction_retention  global_object_recall  oracle_object_recall  predicted_object_recall
           UNet           -2.503471                      0.488372                0.041765              0.786408              0.995146                 0.888350
Attention U-Net           -5.373957                      0.310345               -0.096985              0.854369              0.995146                 0.898058
         UNet++           -4.796426                      0.500000                0.090420              0.839806              0.995146                 0.917476
     DeepLabV3+           -1.554257                      0.387755                0.183829              0.762136              1.000000                 0.854369
[rows shown 4 / 4]


## 10. Global-preservation FP / FN cost

Target recovery must always be read together with the FP / precision cost.

In [10]:
show(summary[['model_display', 'mean_fp_delta', 'mean_fn_delta',
              'mean_global_fn_pixels', 'mean_oracle_fn_pixels', 'mean_predicted_fn_pixels']])

  model_display  mean_fp_delta  mean_fn_delta  mean_global_fn_pixels  mean_oracle_fn_pixels  mean_predicted_fn_pixels
           UNet    7324.970874     -76.689320            2561.621359             725.417476               2484.932039
Attention U-Net    7528.067961     118.825243            2428.135922            1202.941748               2546.961165
         UNet++    7924.495146    -184.135922            2569.378641             532.932039               2385.242718
     DeepLabV3+    8334.417476    -403.359223            2684.252427             490.038835               2280.893204
[rows shown 4 / 4]


## 11. Case improvement / worsening

Exact deterministic comparison with tolerance `1e-12` (**numerical equality handling only — not a gate
threshold**): improved `> +1e-12` · unchanged `<= 1e-12` · worsened `< -1e-12`.

In [11]:
show(summary[['model_display', 'cases_dice_improved', 'cases_dice_unchanged', 'cases_dice_worsened',
              'max_dice_improvement', 'max_dice_worsening']])

  model_display  cases_dice_improved  cases_dice_unchanged  cases_dice_worsened  max_dice_improvement  max_dice_worsening
           UNet                   10                     0                   93              0.386978           -0.789794
Attention U-Net                    5                     0                   98              0.388847           -0.756766
         UNet++                    8                     0                   95              0.381842           -0.799650
     DeepLabV3+                   11                     0                   92              0.393970           -0.750510
[rows shown 4 / 4]


## 12. Component recovery / loss

A component is *detected* iff the binary prediction overlaps the native component by >= 1 pixel.
`GLOBAL_TO_PREDICTED_RECOVERY` = GLOBAL missed AND PREDICTED detected;
`GLOBAL_TO_PREDICTED_LOSS` = GLOBAL detected AND PREDICTED missed.
GLOBAL detection is the Part-1D-B reproduced GLOBAL; the Phase-12 frozen GLOBAL detection flag is also
carried for the ORACLE pairing.

In [12]:
show(summary[['model_display', 'component_recoveries', 'component_losses',
              'empty_recoveries', 'new_empty_predictions']])
print()
print('GLOBAL detection flag agreement with frozen Phase-12:',
      int(comp['global_detection_matches_phase12'].sum()), '/', len(comp))
if not comp['global_detection_matches_phase12'].all():
    d = comp[~comp['global_detection_matches_phase12']]
    print('disagreeing rows (reported, not hidden):')
    show(d[['sample_id', 'native_component_id', 'model', 'size_stratum',
            'global_detected', 'global_detected_phase12']])

  model_display  component_recoveries  component_losses  empty_recoveries  new_empty_predictions
           UNet                    16                 0                 8                      0
Attention U-Net                     9                 0                 3                      0
         UNet++                    14                 0                 5                      0
     DeepLabV3+                    16                 0                11                      0
[rows shown 4 / 4]

GLOBAL detection flag agreement with frozen Phase-12: 608 / 608


## 13. Size-stratified component outcomes

Predeclared strata only: TINY / SMALL / MEDIUM / LARGE.

In [13]:
s = cohort[cohort['cohort_dimension'] == 'size_stratum']
show(s[['cohort_value', 'model_display', 'n_components', 'global_detected', 'oracle_detected',
        'predicted_detected', 'GLOBAL_TO_PREDICTED_RECOVERIES', 'GLOBAL_TO_PREDICTED_LOSSES']])

cohort_value   model_display  n_components  global_detected  oracle_detected  predicted_detected  GLOBAL_TO_PREDICTED_RECOVERIES  GLOBAL_TO_PREDICTED_LOSSES
        TINY            UNet            72               51               71                  60                               9                           0
        TINY Attention U-Net            72               56               71                  60                               4                           0
        TINY          UNet++            72               57               71                  62                               5                           0
        TINY      DeepLabV3+            72               47               72                  55                               8                           0
       SMALL            UNet            28               22               28                  26                               4                           0
       SMALL Attention U-Net            28               2

## 14. hard-Tiny cohort (predeclared)

In [14]:
h = cohort[cohort['cohort_dimension'] == 'hard_tiny']
show(h[['model_display', 'n_components', 'global_detected', 'oracle_detected', 'predicted_detected',
        'GLOBAL_TO_PREDICTED_RECOVERIES', 'GLOBAL_TO_PREDICTED_LOSSES']])

  model_display  n_components  global_detected  oracle_detected  predicted_detected  GLOBAL_TO_PREDICTED_RECOVERIES  GLOBAL_TO_PREDICTED_LOSSES
           UNet            15                0               14                   5                               5                           0
Attention U-Net            15                3               14                   6                               3                           0
         UNet++            15                0               14                   5                               5                           0
     DeepLabV3+            15                2               15                   5                               3                           0
[rows shown 4 / 4]


## 15. global-empty cohort (predeclared)

In [15]:
ge = cohort[cohort['cohort_dimension'] == 'global_empty_image']
show(ge[['model_display', 'n_components', 'global_detected', 'oracle_detected', 'predicted_detected',
        'GLOBAL_TO_PREDICTED_RECOVERIES', 'GLOBAL_TO_PREDICTED_LOSSES']])

  model_display  n_components  global_detected  oracle_detected  predicted_detected  GLOBAL_TO_PREDICTED_RECOVERIES  GLOBAL_TO_PREDICTED_LOSSES
           UNet             8                0                8                   3                               3                           0
Attention U-Net             8                1                8                   4                               3                           0
         UNet++             8                0                8                   3                               3                           0
     DeepLabV3+             8                2                8                   3                               1                           0
[rows shown 4 / 4]


## 16. multi-component cohort (predeclared)

In [16]:
mc = cohort[cohort['cohort_dimension'] == 'multi_component_image']
show(mc[['model_display', 'n_components', 'global_detected', 'oracle_detected', 'predicted_detected',
        'GLOBAL_TO_PREDICTED_RECOVERIES', 'GLOBAL_TO_PREDICTED_LOSSES']])

  model_display  n_components  global_detected  oracle_detected  predicted_detected  GLOBAL_TO_PREDICTED_RECOVERIES  GLOBAL_TO_PREDICTED_LOSSES
           UNet            98               76               97                  87                              11                           0
Attention U-Net            98               78               97                  87                               9                           0
         UNet++            98               77               97                  89                              12                           0
     DeepLabV3+            98               71               98                  84                              13                           0
[rows shown 4 / 4]


## 17. Localization → segmentation linkage

Descriptive association between frozen Part-1C localization coverage and downstream component recovery.
**Not causal proof.** Proposal settings are not tuned here.

In [17]:
show(link[['field', 'value', 'model_display', 'n_components', 'recovered', 'recovery_rate',
           'lost', 'lost_rate']])

                   field  value   model_display  n_components  recovered  recovery_rate  lost  lost_rate
   part1c_bbox_covered90   True            UNet            91          8       0.087912     0        0.0
   part1c_bbox_covered90  False            UNet            61          8       0.131148     0        0.0
   part1c_bbox_covered90   True Attention U-Net            91          3       0.032967     0        0.0
   part1c_bbox_covered90  False Attention U-Net            61          6       0.098361     0        0.0
   part1c_bbox_covered90   True          UNet++            91          4       0.043956     0        0.0
   part1c_bbox_covered90  False          UNet++            61         10       0.163934     0        0.0
   part1c_bbox_covered90   True      DeepLabV3+            91          6       0.065934     0        0.0
   part1c_bbox_covered90  False      DeepLabV3+            61         10       0.163934     0        0.0
   part1c_bbox_covered50   True            UNet        

## 18. Cross-architecture descriptive consistency

Descriptive counts only — **no model ranking, no winner**.

In [18]:
show(cross)
print()
imp = case.pivot_table(index='sample_id', columns='model', values='case_dice_improved', aggfunc='first')
wor = case.pivot_table(index='sample_id', columns='model', values='case_dice_worsened', aggfunc='first')
print('cases where PREDICTED Dice > GLOBAL for all four :', int((imp.sum(axis=1) == 4).sum()))
print('cases non-worse for all four                     :', int((wor.sum(axis=1) == 0).sum()))
print('cases worse for all four                         :', int((wor.sum(axis=1) == 4).sum()))

                            quantity  value
             cases_improved_all_four      3
             cases_nonworse_all_four      3
                cases_worse_all_four     86
components_recovered_by_at_least_one     27
    components_recovered_by_all_four      4
     components_lost_by_at_least_one      0
         components_lost_by_all_four      0
[rows shown 7 / 7]

cases where PREDICTED Dice > GLOBAL for all four : 3
cases non-worse for all four                     : 3
cases worse for all four                         : 86


## 19. 16-case deterministic gallery (REVIEW ONLY)

The gallery is the **frozen Part-1C selection** — not reselected. Predictions for these cases were
preserved during the same Part-1D-B inference execution, so **no second inference pass** was needed.

`GT = EVALUATION / REVIEW ONLY` · `PROPOSALS = GT-FREE FROZEN PART1C`.
**The gallery cannot determine the gate.**

In [19]:
gal = pd.read_csv(TABLES / 'phase13_part1c_gallery_selection.csv')
show(gal)
gdir = ROOT / 'data_cache' / 'phase13_part1d_b_predicted_region_clean_v3' / 'gallery'
print()
print('gallery cases with preserved Part-1D-B outputs:', len([d for d in gdir.iterdir()]) if gdir.exists() else 0)
if gdir.exists():
    print('per-case files:', sorted(p.name for p in (gdir / sorted(d.name for d in gdir.iterdir())[0]).iterdir()))
print()
print('ORACLE visual/prediction artifacts: none regenerated (no oracle rerun authorized).')
print('A valid frozen Phase-12 visual artifact is NOT present in this repository for these 16 cases,')
print('so the ORACLE panel is reported as NOT_AVAILABLE rather than fabricated.')

 selection_rank                        sample_id                                                 sha256_sample_id                                             label
              1 e6db42e46d012311069580771d4bec32 00460fed8885f2ee09f33eb29d955cf76f69db0e939a7e9e7de81e228212721e DETERMINISTIC_PREDICTION_INDEPENDENT_GALLERY_ONLY
              2 938beef9b225180b072bb03cb80a1415 09dc88f718864bf781314bfec698ad1897d2ea1760d7b8a469b1031e6e5d95c7 DETERMINISTIC_PREDICTION_INDEPENDENT_GALLERY_ONLY
              3 501570377c27ca3a1b04f8e78ebbaa45 0b9bc97b046c85418e9a31abe09b6916d9ae430e6f6db051201e6b75df519216 DETERMINISTIC_PREDICTION_INDEPENDENT_GALLERY_ONLY
              4 6f1a73eb1dc3ddb7af81e8a2893a2e76 0c8c198f3a5f956b1c5e6d0cc0d3449cc0ad00b0d7b747ba42970e06d3441e36 DETERMINISTIC_PREDICTION_INDEPENDENT_GALLERY_ONLY
              5 66b947d8ead11ae1260bc3142ca1707d 0c9284c277704d6cea19ce63c0631a3216f4cd261108f23068812e481a77ecb6 DETERMINISTIC_PREDICTION_INDEPENDENT_GALLERY_ONLY
              6 

## 20. Limitations

- **Oracle is an upper bound, not deployable.** Phase-12 oracle geometry uses GT-derived component
  locations; the predicted path uses frozen GT-free Part-1C proposals at margin 0.20 (oracle used 0.25).
  `PRACTICAL_GT_FREE_ORACLE_BENEFIT_RETENTION` is therefore **not** a pure mechanism-retention measure.
- Single seed (42) per architecture; no paired significance testing was performed, and none is claimed.
- Retention quantities are descriptive; no numerical gate threshold exists by design.
- Component detection uses a >= 1 native pixel overlap rule under 8-connectivity, applied to the frozen
  Phase-12 component identities.
- HD95 invalid entries are explicitly labelled with their reason rather than fabricated.
- `GLOBAL_NATIVE` in this phase is a Part-1D-B reproduction (verified to <= 1e-4 max abs case Dice);
  the ORACLE comparator is the frozen Phase-12 authoritative table.
- The gallery is review-only and carries no gate authority.

## 21. Explicit gate boundary

| item | state |
|---|---|
| `GO` | **NOT ISSUED** |
| `REVISE_LOCALIZER` | **NOT ISSUED** |
| `NO_GO` | **NOT ISSUED** |
| `REGIONAL_INTEGRATION_GATE_AFTER_PHASE13` | **NOT_DECIDED** |
| regional mechanism promoted | **NO** |
| Phase 14 | **NOT_STARTED** |
| TEST reads | **0** |

The regional-integration gate belongs to a separate supervisor-reviewed **Part 1E-B**. Retrofit
criteria after results are **forbidden**.

In [20]:
print('gate_state                        :', gate['gate_state'])
print('gate verdict issued by Part-1D-B  : NOT_ISSUED')
print('regional mechanism promoted       : NO')
print('Part1E-B gate adjudication        :', completion['continuity']['phase13_part1e_b_gate_adjudication'])
print('Phase14                           :', completion['continuity']['phase14'])
print('TEST reads                        :', completion['counters']['test_reads'])

gate_state                        : NOT_DECIDED
gate verdict issued by Part-1D-B  : NOT_ISSUED
regional mechanism promoted       : NO
Part1E-B gate adjudication        : ELIGIBLE_FOR_SUPERVISOR_AUTHORIZATION
Phase14                           : NOT_STARTED
TEST reads                        : 0


## 22. Artifact SHA registry

In [21]:
reg = {
    'phase13_part1d_predicted_region_protocol_frozen_r1.json': AUDITS / 'phase13_part1d_predicted_region_protocol_frozen_r1.json',
    'phase13_regional_integration_gate_criteria_frozen.json': AUDITS / 'phase13_regional_integration_gate_criteria_frozen.json',
    'phase13_part1d_b_inference_start.json': AUDITS / 'phase13_part1d_b_inference_start.json',
    'phase13_part1d_b_global_reproduction.json': AUDITS / 'phase13_part1d_b_global_reproduction.json',
    'phase13_part1d_b_predicted_region_evaluation.json': AUDITS / 'phase13_part1d_b_predicted_region_evaluation.json',
    'phase13_part1d_predicted_region_case_model_metrics.csv': TABLES / 'phase13_part1d_predicted_region_case_model_metrics.csv',
    'phase13_part1d_predicted_region_component_model_metrics.csv': TABLES / 'phase13_part1d_predicted_region_component_model_metrics.csv',
    'phase13_part1d_predicted_region_model_summary.csv': TABLES / 'phase13_part1d_predicted_region_model_summary.csv',
    'phase13_part1d_b_global_reproduction_gate.csv': TABLES / 'phase13_part1d_b_global_reproduction_gate.csv',
    'phase13_part1d_cross_architecture_summary.csv': TABLES / 'phase13_part1d_cross_architecture_summary.csv',
    'phase13_part1d_predicted_region_cohort_summary.csv': TABLES / 'phase13_part1d_predicted_region_cohort_summary.csv',
    'phase13_part1d_localization_segmentation_linkage.csv': TABLES / 'phase13_part1d_localization_segmentation_linkage.csv',
}
for k, p in reg.items():
    print(f'{sha256_file(p)}  {k}' if p.exists() else f'MISSING  {k}')

9089b948ef7563252963bf68d0e0b57a9e891c5e913f54a315bce94c98207eb8  phase13_part1d_predicted_region_protocol_frozen_r1.json
342a8bcac2c87bf0b85f7bbd963179e74f029825c0434cf527fbd939d2d13c21  phase13_regional_integration_gate_criteria_frozen.json
ec34a18596182cc30690b42bf8d1f4102774c9ef48a9fa3563dbf55b7881155e  phase13_part1d_b_inference_start.json
a2914fb7c67cb68c8b60074fbf52fc1b424707201af65e60aee7c36a8680b82f  phase13_part1d_b_global_reproduction.json
365b5ad80215a7cc6143d255f7b4ec0b97e6bf6f8de8be65826239692d8c2847  phase13_part1d_b_predicted_region_evaluation.json
53e93982c0235e32568e01fbddbdd858d205077e3e0a50fd69bef992454f2658  phase13_part1d_predicted_region_case_model_metrics.csv
f82e0632898467d32916eb0b71a9fff75818548665477052e2db9b365d313c9d  phase13_part1d_predicted_region_component_model_metrics.csv
6f9391f20233854c2f11f9002b7b1bc03dbcfc420dc22779daaaab11c3f697ba  phase13_part1d_predicted_region_model_summary.csv
ec619e0af0ed924c4b609e567a045f9b852d8cd95e112331093d08515006a215  

## Appendix — complete frozen tables

Deterministic text rendering of the complete frozen tables (all rows visible). The CSV artifacts remain
the machine-readable authority.

In [22]:
print('=' * 100)
print('MODEL SUMMARY — 4 / 4 rows')
print('=' * 100)
print(summary.to_string(index=False))
print()
print('=' * 100)
print('CASE-MODEL TABLE — %d / %d rows' % (len(case), len(case)))
print('=' * 100)
cc = case[['sample_id', 'model_display', 'global_dice', 'oracle_dice', 'predicted_dice',
           'predicted_minus_global_dice', 'oracle_minus_global_dice', 'predicted_minus_oracle_dice',
           'proposal_count', 'proposal_union_area_fraction']]
print(cc.to_string(index=False))

MODEL SUMMARY — 4 / 4 rows
         model   model_display  global_macro_dice  global_macro_iou  global_macro_precision  global_macro_recall  oracle_macro_dice  oracle_macro_iou  oracle_macro_precision  oracle_macro_recall  predicted_macro_dice  predicted_macro_iou  predicted_macro_precision  predicted_macro_recall  DELTA_ORACLE_DICE  DELTA_PRED_DICE  RETENTION_DICE  ORACLE_GAP_DICE  iou_gain_retention  global_object_recall  oracle_object_recall  predicted_object_recall  object_recall_gain_retention  mean_global_fn_pixels  mean_oracle_fn_pixels  mean_predicted_fn_pixels  fn_reduction_retention  mean_fp_delta  mean_fn_delta  cases_dice_improved  cases_dice_unchanged  cases_dice_worsened  max_dice_improvement  max_dice_worsening  component_recoveries  component_losses  empty_recoveries  new_empty_predictions
          unet            UNet           0.594180          0.476100                0.693919             0.653931           0.724722          0.586966                0.670271          

In [23]:
print('=' * 100)
print('COMPONENT-MODEL TABLE — %d / %d rows' % (len(comp), len(comp)))
print('=' * 100)
cp = comp[['sample_id', 'native_component_id', 'model_display', 'size_stratum',
           'global_detected', 'oracle_detected', 'predicted_detected',
           'GLOBAL_TO_PREDICTED_RECOVERY', 'GLOBAL_TO_PREDICTED_LOSS',
           'part1c_bbox_covered90', 'part1c_pixel_covered90']]
print(cp.to_string(index=False))

COMPONENT-MODEL TABLE — 608 / 608 rows
                       sample_id  native_component_id   model_display size_stratum  global_detected  oracle_detected  predicted_detected  GLOBAL_TO_PREDICTED_RECOVERY  GLOBAL_TO_PREDICTED_LOSS  part1c_bbox_covered90  part1c_pixel_covered90
66b947d8ead11ae1260bc3142ca1707d                    1            UNet         TINY            False             True                True                          True                     False                   True                    True
66b947d8ead11ae1260bc3142ca1707d                    1 Attention U-Net         TINY             True             True                True                         False                     False                   True                    True
66b947d8ead11ae1260bc3142ca1707d                    1          UNet++         TINY             True             True                True                         False                     False                   True                    True
6

In [24]:
print('READ-ONLY notebook — no model construction, no CUDA request, no inference, no TEST access.')
print('case rows  :', len(case))
print('comp rows  :', len(comp))
print('summary    :', len(summary))

READ-ONLY notebook — no model construction, no CUDA request, no inference, no TEST access.
case rows  : 412
comp rows  : 608
summary    : 4
